# Bagging-based feature selection

This notebook finds a feature set independently for every power-quality disturbance (PQD) class. It follows the idea in `Vossi_01_Task.ipynb`: start with all eight features and remove features that do not improve classification, while evaluating the decision across cross-validation folds and multiple classifiers.

The evaluation uses stratified k-fold cross-validation. Each candidate feature set is trained and validated across several folds, while the scaler is fitted only on each fold's training data. Each class is evaluated as a one-vs-rest binary problem, so the output preserves class-specific feature choices.


## 1. Imports and paths

The helper module contains reusable data-loading and feature-selection functions. The path setup makes the notebook work both from `Code/vossi` and from the repository root.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
CODE_DIR = next(
    (path / "Code" for path in [CURRENT_DIR, *CURRENT_DIR.parents]
     if (path / "Code" / "data").is_dir()),
    Path(__file__).resolve().parents[2],
)
TASK_DIR = CODE_DIR / "01 Task"
VOSSI_DIR = TASK_DIR / "vossi"
DATA_DIR = CODE_DIR / "data"
sys.path.insert(0, str(VOSSI_DIR))

from bagging_feature_selection import (
    load_dataset,
    majority_vote_features,
    run_feature_selection,
)


## 2. Load and inspect the data

Each CSV file represents one disturbance class; its filename supplies the class label. The loader standardizes the columns to `F1` through `F8`, adds the source noise level, and combines all requested SNR folders. The noise-level column is retained for diagnostics but is not passed to the classifiers.


In [ ]:
folders = [DATA_DIR / name for name in (
    "SNR_20db", "SNR_30db", "SNR_40db", "SNR_50db", "SNR_noiseless"
)]
data = load_dataset(folders)
print(f"Loaded {len(data):,} samples from {data['noise_level'].nunique()} noise levels")
data.head()


## 3. Select features for every class and classifier

For each class and classifier pair, the selector creates a one-vs-rest target: the selected class is positive and all other classes are negative. It starts with all eight features, evaluates every possible one-feature removal using stratified k-fold validation F1, accepts the best removal when it is not meaningfully worse than the current set, and repeats until no further removal is supported or one feature remains.

The scaler is inside the estimator pipeline, so it is fitted separately for every cross-validation training fold. This prevents validation-fold information from leaking into preprocessing. `min_improvement=0.002` prevents tiny score fluctuations from causing unstable removals. Increase `n_splits` for a final experiment.


In [ ]:
results = run_feature_selection(
    data,
    n_splits=5,
    min_improvement=0.002,
    random_state=42,
)

# The function returns one list containing every class/classifier result.
assert isinstance(results, list), (
    "Restart the kernel to load the updated feature-selection module."
)
print(f"Generated {len(results)} class/classifier feature selections")

summary = pd.DataFrame({
    "target_label": [result.target_label for result in results],
    "classifier": [result.classifier for result in results],
    "selected_features": [list(result.selected_features) for result in results],
    "cross_validation_f1": [result.score for result in results],
})
summary


## 4. Inspect the per-class recommendations

The table contains one row for every class and classifier, so selected features can differ between disturbance types. The majority vote below combines only the classifier results for the same class. A feature must be selected by a strict majority of classifiers for that class to be recommended.


In [ ]:
for target_label in summary["target_label"].unique():
    print(f"\n{target_label}")
    display(summary[summary["target_label"] == target_label])

recommended_features = majority_vote_features(results)
recommendation_table = pd.DataFrame({
    "target_label": list(recommended_features),
    "recommended_features": [
        list(features) for features in recommended_features.values()
    ],
})
recommendation_table


## 5. Interpret the output

The summary reports the selected subset and mean cross-validation one-vs-rest F1 for each classifier and target class. `recommended_features` is the final per-class output: each entry is the feature set selected by a strict majority of classifiers for that class. A high F1 score means that the selected class can be separated from the other disturbance types using that subset.

Use `recommended_features["Sag"]` (for example) when building a class-specific downstream model. If the classifiers disagree completely, the helper returns all tied most-frequently selected features so every class still receives a usable recommendation. For a final reported result, rerun with a larger `n_splits` value and keep the random seed fixed so the experiment remains reproducible.
